In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
from scipy import stats

input_df = pd.read_pickle(snakemake.input[0])

In [ ]:
region_list = [
    'Left-Cerebral-White-Matter', 'Right-Cerebral-White-Matter', 
    'frontal_left_wm', 'frontal_right_wm', 
    'temporal_left_wm', 'temporal_right_wm',
    'parietal_left_wm', 'parietal_right_wm',
    'occipital_left_wm', 'occipital_right_wm',
    'cerebellum_left', 'cerebellum_right'
              ]
#filter df to include only selected regions, with 90% certainty of white matter, and outliers removed
filtered_df = input_df.query('region in @region_list and outliers_removed == True and segmentation == "mni_icbm152_nlin_asym_09c_wm90percent_lobes"')
#split contrast into separate contrast and b1corr T/F columns
filtered_df['b1corr'] = filtered_df['contrast'].str.contains('b1corr')
filtered_df['contrast'] = filtered_df['contrast'].str.replace('_b1corr', '')
#split acquisition into separate acquisition and Vref_lessthan100 T/F columns
filtered_df['Vref_lessthan100'] = filtered_df['acquisition'].str.contains("(?i)Vref(0*(?:[1-9][0-9]?|100))", regex=True)
filtered_df['acquisition'] = filtered_df['acquisition'].str.replace("(?i)Vref(0*(?:[1-9][0-9]?|100))", "", regex=True)

In [ ]:
#for calculating pearson, split Vref100 and Vref less than 100 into two separate dataframes
#rename mean to Vref100 or Vref_lessthan100 accordingly
#then merge the two datasets, such that the means for Vref100 and Vref_lessthan100 now have separate columns
columns = ['subject', 'session', 'field_strength', 'modality', 'acquisition', 'contrast', 'b1corr', 'region', 'mean']
df_vref_false = filtered_df[filtered_df['Vref_lessthan100'] == False][columns].rename(
    {'mean':'Vref100'}, axis=1)
df_vref_true = filtered_df[filtered_df['Vref_lessthan100'] == True][columns].rename(
    {'mean':'Vref_lessthan100'}, axis=1)

keys = ['subject', 'session', 'field_strength', 'modality', 'acquisition', 'contrast', 'b1corr', 'region']
df_by_vref = df_vref_false.merge(
    df_vref_true, on=keys, how='inner')

In [ ]:
#calculate within-subject pearson r for Vref100 vs Vref_lessthan100 between regions
#for each subject, session, field strength, modality, acquisition, contrast, and b1corr
indices = ['subject', 'session', 'field_strength', 'modality', 'acquisition', 'contrast', 'b1corr']
pearson_vref = df_by_vref.groupby(indices).apply(
    lambda d: pg.pairwise_corr(
        data=d, columns=['Vref100', 'Vref_lessthan100']
    )).reset_index()
pearson_vref['test'] = 'Vref'
pearson_vref = pearson_vref.rename({'r':'pearson_r','CI95':'pearson_CI95'}, axis=1)[[
    'test', 'subject', 'session', 'field_strength', 'modality', 'contrast', 'b1corr', 'acquisition', 'pearson_r', 'pearson_CI95'
]].reset_index(drop=True)

In [ ]:
#save to csv
pearson_vref.to_csv(snakemake.output.vref)

In [ ]:
pearson_vref

In [ ]:
#for pearson r, create separate b1corr and b1uncorr dataframes for Vref100
#rename the Vref100 values (aka the means) to b1corr_false or b1corr_true
#then merge back into one dataframe, where b1corr_false and b1corr_true values have their own columns
columns = ['subject', 'session', 'field_strength', 'modality', 'acquisition', 'contrast', 'region', 'Vref100']
df_b1corr_false = df_vref_false[filtered_df['b1corr'] == False][columns].rename(
    {'Vref100':'b1corr_false'}, axis=1)
df_b1corr_true = df_vref_false[filtered_df['b1corr'] == True][columns].rename(
    {'Vref100':'b1corr_true'}, axis=1)

keys = ['subject', 'session', 'field_strength', 'modality', 'acquisition', 'contrast', 'region']
df_by_b1corr = df_b1corr_false.merge(
    df_b1corr_true, on=keys, how='inner')

In [ ]:
#calculate within-subject pearson r of b1corr vs b1uncorr for Vref100 across regions
#for each subject, session, field strength, modality, acquisiton, and contrast
indices = ['subject', 'session', 'field_strength', 'modality', 'acquisition', 'contrast']
pearson_b1corr = df_by_b1corr.groupby(indices).apply(
    lambda d: pg.pairwise_corr(
        data=d, columns=['b1corr_false', 'b1corr_true']
    )).reset_index()
pearson_b1corr['test'] = 'b1corr'
pearson_b1corr = pearson_b1corr.rename({'r':'pearson_r','CI95':'pearson_CI95'}, axis=1)[[
    'test', 'subject', 'session', 'field_strength', 'modality', 'contrast', 'acquisition', 'pearson_r', 'pearson_CI95'
]].reset_index(drop=True)

In [ ]:
#save to csv
pearson_b1corr.to_csv(snakemake.output.b1corr)

In [ ]:
pearson_b1corr

In [ ]:
#Use pearson r as a measure of symmetry (ICC not appropriate for this use case)
#Calculate symmetry of Vref100 for comparing symmetry of b1corr vs b1uncorr
df_left = df_vref_false[df_vref_false['region'].str.contains('left')].rename(
    {'Vref100':'left'}, axis=1)
df_left['region'] = df_left['region'].str.replace("_left", "")
df_right = df_vref_false[df_vref_false['region'].str.contains('right')].rename(
    {'Vref100':'right'}, axis=1)
df_right['region'] = df_right['region'].str.replace("_right", "")

keys = ['subject', 'session', 'field_strength', 'modality', 'acquisition', 'contrast', 'region', 'b1corr']
df_leftright = df_left.merge(
    df_right, on=keys, how='inner')

indices = ['subject', 'session', 'field_strength', 'modality', 'acquisition', 'contrast', 'b1corr']
pearson_symmetry = df_leftright.groupby(indices).apply(
    lambda d: pg.pairwise_corr(
        data=d, columns=['left', 'right']
    )).reset_index()
pearson_symmetry['test'] = 'symmetry'
pearson_symmetry = pearson_symmetry.rename({'r':'pearson_r','CI95':'pearson_CI95'}, axis=1)[[
    'test', 'subject', 'session', 'field_strength', 'modality', 'acquisition', 'contrast', 'b1corr', 'pearson_r', 'pearson_CI95'
]].reset_index(drop=True)

In [ ]:
pearson_symmetry.to_csv(snakemake.output.sym)

In [ ]:
pearson_symmetry